vost 무료 모델

In [10]:
!pip install vosk sounddevice requests

In [11]:
# ─── 2) 모델 다운로드 및 압축 해제 ────────────────────────────
import requests, zipfile, io, os

URL = "https://alphacephei.com/vosk/models/vosk-model-small-ko-0.22.zip"
MODEL_DIR = "vosk-model-small-ko-0.22"

# 이미 받아둔 경우 건너뛰기
if not os.path.exists(MODEL_DIR):
    print("Downloading Vosk model...")
    resp = requests.get(URL, stream=True)
    resp.raise_for_status()
    
    print("Extracting model...")
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        z.extractall()
    
    print(f"✅ '{MODEL_DIR}' 폴더가 생성되었습니다.")
else:
    print(f"'{MODEL_DIR}' 폴더가 이미 존재합니다.")
    
# 확인
print("Contents of current dir:", os.listdir())


Extracting model...
✅ 'vosk-model-small-ko-0.22' 폴더가 생성되었습니다.
Contents of current dir: ['.git', 'huggingface.ipynb', 'kold_token_labels.csv', 'kold_v1(1).csv', 'README.md', 'realtime_stt.ipynb', 'TM_model_ver1.ipynb', 'TM_model_ver2.ipynb', 'TM_model_ver3.ipynb', 'vosk-model-small-ko-0.22', '텍스트마이닝 팀프로젝트의 사본.ipynb']


In [12]:
from vosk import Model, KaldiRecognizer

model = Model("vosk-model-small-ko-0.22")

In [1]:
import sys, queue, json
import sounddevice as sd
from vosk import Model, KaldiRecognizer

# ——— 설정 ———
MODEL_PATH = "vosk-model-small-ko-0.22"  # 압축 해제한 모델 디렉터리
SAMPLE_RATE = 16000                     # 모델이 기대하는 샘플링 레이트

# 마이크 입력을 받을 큐
q = queue.Queue()

def audio_callback(indata, frames, time, status):
    if status:
        print(f"Audio status: {status}", file=sys.stderr)
    q.put(bytes(indata))

# 모델 로드
model = Model(MODEL_PATH)
rec = KaldiRecognizer(model, SAMPLE_RATE)

# 마이크 스트림 열기
with sd.RawInputStream(samplerate=SAMPLE_RATE, blocksize=8000, dtype='int16',
                       channels=1, callback=audio_callback):
    print("===== 실시간 STT 시작 =====\n(CTRL+C로 종료)\n")
    while True:
        data = q.get()
        if rec.AcceptWaveform(data):
            # 완성된 문장
            result = json.loads(rec.Result())
            print(f"\n▶ 최종: {result.get('text','')}\n")
        else:
            # 중간 인식 결과
            partial = json.loads(rec.PartialResult())
            print(f"… {partial.get('partial','')}", end='\r')


===== 실시간 STT 시작 =====
(CTRL+C로 종료)

… 는 장면이 시즌이 제 를 내세워 문호
▶ 최종: 는 장에는 시즌이 넷 [29] 문호

… 쓰시 는 낮은 A 이 (백) 이 (십) 일 [쩜] [이] 지지층 BGM 이 제 [14] 호 와 G S
▶ 최종: 쓰시 는 낮은 A 이 (백) 이 (십) 일 [쩜] 이 지지층 비즈 이지은 이 제 [14] 호 와 지령 는 겠지

… IS 긴 스
▶ 최종: IS 긴 스

… 
▶ 최종: 



KeyboardInterrupt: 

faster-whisper library

In [15]:
pip install faster-whisper sounddevice


  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
  Using cached pyreadline3-3.5.4-py3-none-any.whl.metadata (4.7 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 11.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   --- ------------------------------------ 2.4/27.9 MB 12.2 MB/s eta 0:00:03
   ------ --------------------------------- 4.7/27.9 MB 11.9 MB/s eta 0:00:02
   ---------- ----------------------------- 7.1/27.9 MB 11.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.7/27.9 MB 11.8 MB/s eta 0:00:02
   ----------------- ---------------------- 12.3/27.9 MB 11.9 MB/s eta 0:00:02
   --------------------- ------------------ 14.7/27.9 MB 11.7 MB/s eta 0:00:02
   ------------------------ --------------- 17.0/27.9 MB 11.7 MB/s eta 0:00:01
   ------------------

In [2]:
import sounddevice as sd

print("— 입력 채널이 있는 장치만 표시 —")
for idx, dev in enumerate(sd.query_devices()):
    if dev['max_input_channels'] > 0:
        print(f"{idx}: {dev['name']}  (입력 채널: {dev['max_input_channels']})")

— 입력 채널이 있는 장치만 표시 —
0: Microsoft 사운드 매퍼 - Input  (입력 채널: 2)
1: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
2: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
6: 주 사운드 캡처 드라이버  (입력 채널: 2)
7: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
8: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
14: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
15: 마이크(Synaptics SmartAudio HD)  (입력 채널: 2)
16: Microphone (Conexant HD Audio capture)  (입력 채널: 2)
17: Microphone (Conexant HD Audio capture)  (입력 채널: 2)
23: 헤드셋 마이크 (@System32\drivers\bthhfenum.sys,#2;%1 Hands-Free%0
;(문현검색2-2))  (입력 채널: 1)
25: 머리에 거는 수화기 (@System32\drivers\bthhfenum.sys,#2;%1 Hands-Free%0
;(Galaxy Buds3 (1FF4)))  (입력 채널: 1)


In [ ]:
from faster_whisper import WhisperModel
import sounddevice as sd
import numpy as np
import queue

# ─── 모델 로드 ───────────────────────────────
model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

# ─── 스트리밍 세팅 ───────────────────────────
SAMPLE_RATE = 16000
CHUNK_SEC   = 1.5            # 1.5초 단위로 청크
CHUNK_SIZE  = int(SAMPLE_RATE * CHUNK_SEC)
BLOCKSIZE   = 1024

q = queue.Queue()
buffer = np.zeros((0,), dtype=np.float32)

def callback(indata, frames, time, status):
    # 들어오는 파형을 큐에 넣기
    q.put(indata[:, 0].copy())

# ─── 스트림 열기 ─────────────────────────────
print("===== STT 스트리밍 시작 (Ctrl+C로 종료) =====")
with sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
    blocksize=BLOCKSIZE,
    callback=callback
):
    while True:
        frame = q.get()
        buffer = np.concatenate([buffer, frame])

        # 1. 버퍼에 CHUNK_SEC 이상 쌓이면
        if len(buffer) >= CHUNK_SIZE:
            # 2. 디버그: 버퍼에 실제 음성 신호가 있는지 진폭 확인
            print(f"[DEBUG] amp: {buffer.min():.3f} … {buffer.max():.3f}")

            # 3. 변환 (VAD 필터 끔)
            segments, info = model.transcribe(
                buffer,
                beam_size=5,
                vad_filter=False
            )

            # 4. 결과 출력
            for seg in segments:
                print(f"{seg.start:.1f}s–{seg.end:.1f}s ▶ {seg.text}")

            # 5. 버퍼 초기화: 마지막 0.5초만 남겨두기(중첩)
            keep = int(SAMPLE_RATE * 0.5)
            buffer = buffer[-keep:]

C:\Users\dasolkim7\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===== STT 스트리밍 시작 (Ctrl+C로 종료) =====
[DEBUG] amp: -0.639 … 0.524
0.0s–2.0s ▶  Thank you very much.
[DEBUG] amp: -0.528 … 0.343


google cloud stt service

In [2]:
pip install google-cloud-speech sounddevice

Note: you may need to restart the kernel to use updated packages.


In [6]:
# 1) os.environ 사용
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"textmining-461305-f0cddebc86fe.json"

In [15]:
import queue, time
import sounddevice as sd
from google.cloud import speech
from google.api_core.exceptions import OutOfRange

# ——— 설정 ——————————————————————————
CLIENT = speech.SpeechClient.from_service_account_file(
    r"textmining-461305-f0cddebc86fe.json"
)
CONFIG = speech.RecognitionConfig(
    encoding        = speech.RecognitionConfig.AudioEncoding.LINEAR16,
    sample_rate_hertz = 16000,
    language_code     = "ko-KR",
)
STREAMING_CONFIG = speech.StreamingRecognitionConfig(
    config          = CONFIG,
    interim_results = True,
    single_utterance=False,
)

SAMPLE_RATE         = 16000
BLOCKSIZE           = 1024
KEEPALIVE_INTERVAL  = 0.5  # 빈 청크 전송 주기
q = queue.Queue()

def audio_callback(indata, frames, time_, status):
    if status:
        print("Audio status:", status)
    # indata는 cffi 버퍼이므로 bytes()로 변환
    q.put(bytes(indata))

def request_generator():
    while True:
        try:
            chunk = q.get(timeout=KEEPALIVE_INTERVAL)
            yield speech.StreamingRecognizeRequest(audio_content=chunk)
        except queue.Empty:
            yield speech.StreamingRecognizeRequest(audio_content=b"")

def start_streaming():
    while True:
        print("▶ 스트리밍 시작")
        try:
            responses = CLIENT.streaming_recognize(
                STREAMING_CONFIG,  # config는 여기서만!
                request_generator()
            )
            for resp in responses:
                for result in resp.results:
                    tag = "[Final]" if result.is_final else "[Interim]"
                    print(f"{tag} {result.alternatives[0].transcript}")
        except OutOfRange:
            print("⚠️ Audio timeout, 재연결 시도…")
            continue
        except Exception as e:
            print("❗ 스트리밍 오류:", e)
            break

# ——— 마이크 스트림 열고 시작 ——————————————————
with sd.RawInputStream(
    samplerate=SAMPLE_RATE,
    blocksize=BLOCKSIZE,
    dtype='int16',
    channels=1,
    callback=audio_callback
):
    start_streaming()


▶ 스트리밍 시작
[Interim] 하트가
[Interim] 1시간
[Interim] 1시간에
[Interim] 1시간에
[Interim] 1시간에
[Interim]  컨셉
[Interim] 1시간에
[Interim]  컨셉은
[Interim] 1시간에 컨셉은
[Interim] 하트가 4 1 7 3 최저가
[Interim] 하트가 4 1 7 3 최적화
[Interim] 하트가 4 1 7 3 최저가
[Interim] 하트가 4 1 7 3
[Interim]  최저가
[Interim] 하트가 4 1 7 3 최저가
[Interim] 하트가 4 1 7 3 최저가
[Interim]  머신러닝
[Interim] 하트가 4 1 7 3 최저가 머신러닝
[Interim] 하트가 4 1 7 3 최저가
[Interim]  머신 러닝 계속하기
[Interim] 하트가 4 1 7 3 최저가
[Interim]  머신러닝 최저가가
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가
[Interim]  성능이라
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가
[Interim]  성능이란
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가
[Interim]  성능이란 컨셉
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가
[Interim]  성능이란 선택된
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가 성능이란
[Interim]  컨셉때문에
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가 성능이란
[Interim]  컨셉때문에 무적
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가 성능이란
[Interim]  컨셉때문에 무조건
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가 성능이란 컨셉때문에
[Interim]  무조건
[Interim] 하트가 4 1 7 3 최저가 머신러닝 최저가가 성능이란 컨셉때문에
[Int

KeyboardInterrupt: 

네이버 클로바 realtime stt

https://jaey0ng.tistory.com/57

https://www.ncloud.com/product/aiService/csr

https://velog.io/@team_vino/4%EB%B2%88

In [ ]:
import websocket
import threading
import sounddevice as sd
import numpy as np
import json
import base64

# 네이버 클라우드에서 발급받은 인증 정보
API_KEY = "YOUR_CLIENT_ID"
API_SECRET = "YOUR_CLIENT_SECRET"
LANG = "Kor"

# WebSocket URL
ws_url = f"wss://clovaspeech-gw.ncloud.com/ws/v1/recognize?lang={LANG}"

def on_message(ws, message):
    data = json.loads(message)
    print("Recognized:", data.get("text", ""))

def on_error(ws, error):
    print("Error:", error)

def on_close(ws, close_status_code, close_msg):
    print("WebSocket closed")

def on_open(ws):
    def run(*args):
        # 마이크에서 16kHz, 16bit, mono로 0.5초씩 읽어서 전송
        with sd.InputStream(samplerate=16000, channels=1, dtype='int16') as stream:
            print("Start speaking...")
            while True:
                audio_chunk, _ = stream.read(int(16000 * 0.5))  # 0.5초 분량
                audio_bytes = audio_chunk.tobytes()
                # base64 인코딩
                audio_b64 = base64.b64encode(audio_bytes).decode('utf-8')
                # 데이터 전송
                ws.send(json.dumps({
                    "accessKey": API_KEY,
                    "secretKey": API_SECRET,
                    "lang": LANG,
                    "audioContent": audio_b64,
                    "format": "PCM",
                    "sampleRate": 16000,
                }))
    threading.Thread(target=run).start()

if __name__ == "__main__":
    ws = websocket.WebSocketApp(ws_url,
                                on_open=on_open,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close)
    ws.run_forever()
